# 01. What to train on: Kaggle sentences, synthetic posts, or both?

We have two kinds of labeled text:

- **Kaggle sentences** (63,388 in `data/cv_rows.parquet`, 3 folds): plenty of them, but short sentences
  from news articles, labeled without street-type words ("Хрещатик", not "вулиці Хрещатик").
- **Synthetic posts** (5,839, `synthetic/`): look like the test posts (Telegram style, emoji, line
  breaks, addresses) and follow the test labeling style, but there are far fewer and they are written by
  an LLM.

The adversarial check showed that XLM-R separates sentences from test posts almost perfectly (AUC 0.99),
and synthetic posts less well (0.94). So we expect the synthetic posts to matter most for the score on
posts, and the sentences to add variety of place names. This notebook finds the mix.

**Fixed for every run:** XLM-R base, AdamW, lr 3e-5 with linear decay and 10% warmup, 3 epochs, batch 32,
windows of 256 tokens. Validation fold 0: the model never sees fold 0 of the sentences or of the
synthetic posts.

**Scores** (see `ner/evaluate.py`): `cv` = held-out sentences (comparable with Assignment 3), `synthetic`
= held-out synthetic posts, `half_B` = test posts with pseudo-labels that the generator never saw (our
main score), `half_A` = test posts the generator saw, `gold26` = the 26 human-labeled posts. After
submitting a run to Kaggle, add its public score to `transformers/leaderboard.csv`.

### Setup

On Colab (T4 or better): clones the repo, installs W&B and keeps every run on Drive in
`dl-a4-ner/runs`, so a disconnect loses at most the run in progress (finished runs are loaded, not
retrained). Teammates: put a shortcut to the shared `dl-a4-ner` folder in your My Drive, so all runs
land in one place. Set `WANDB_ENTITY` to the team's W&B entity to log into one shared project.
Locally the notebook runs from the repo root and writes to `transformers/runs`. To check the whole
notebook in a few minutes before using GPU time, set the environment variable `NER_SMOKE=1`: every run
shrinks to a few hundred texts and goes to `transformers/runs_smoke`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
WANDB_PROJECT = "dl-a4-ner"  # None switches W&B off
WANDB_ENTITY = None  # the team's W&B entity, so all teammates log into one project

if "google.colab" in sys.modules:
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/dl-a4-ner/runs")
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    OUT = Path("transformers/runs")
sys.path.insert(0, os.getcwd())
if os.environ.get("NER_SMOKE"):  # quick end-to-end check with tiny runs, see ner.train.smoke
    OUT, WANDB_PROJECT = Path("transformers/runs_smoke"), None
OUT.mkdir(parents=True, exist_ok=True)
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_PROJECT:
    import wandb

    wandb.login()

In [ ]:
import dataclasses

from ner import plots
from ner.data import DataSpec
from ner.train import RunConfig, load_run, run_or_load

BASE = RunConfig(name="base", model="FacebookAI/xlm-roberta-base", fold=0, tags=("01_data_mix",))


def train(name, **changes):
    cfg = dataclasses.replace(BASE, name=f"01-{name}", **changes)
    return run_or_load(cfg, OUT, WANDB_PROJECT)

## Each source alone

The two baselines. Sentences only is what Assignment 3 trained on, so its `cv` score is directly
comparable with the RNNs; the gap between `cv` and `half_B` shows the domain shift. Synthetic only shows
how far the 3,900 training posts of fold 0 get without any Kaggle data.

In [ ]:
train("sentences", data=DataSpec(synthetic="none"))

In [ ]:
train("synthetic", data=DataSpec(sentences=0))

## Mixes

All 42k training sentences outnumber the synthetic posts about 11 to 1, so the model mostly learns the
sentence style. We try three ways to give the posts more weight: fewer sentences (a random 10k sample),
repeating every synthetic post 3 times per epoch, and two stages (first sentences, then posts only,
with a lower learning rate, so the last thing the model learns is the post style and labeling rules).

In [ ]:
train("sent10k-syn", data=DataSpec(sentences=10_000))

In [ ]:
train("sentall-syn", data=DataSpec())

In [ ]:
train("sentall-syn3", data=DataSpec(synthetic_repeat=3))

In [ ]:
train(
    "two-stage",
    data=DataSpec(synthetic="none"),
    epochs=2,
    stage2_data=DataSpec(sentences=0),
    stage2_epochs=2,
    stage2_lr=2e-5,
)

## Filtering the synthetic posts by the adversarial score

`synthetic_adversarial.ipynb` gave every synthetic post a test-likeness percentile. Here we keep only
the more test-like half, or keep all posts but weight each post's loss by its percentile (mean weight 1).
Set `MIX` to the best mix from above before running these (default: 10k sentences).

In [ ]:
MIX = DataSpec(sentences=10_000)

train("syn-top50", data=dataclasses.replace(MIX, synthetic="top50"))

In [ ]:
train("syn-weighted", data=dataclasses.replace(MIX, weight_by_test_like=True))

## Results

Runs that are not finished yet (or were run by a teammate into another folder) are skipped.

In [ ]:
NAMES = ["sentences", "synthetic", "sent10k-syn", "sentall-syn", "sentall-syn3", "two-stage", "syn-top50", "syn-weighted"]
runs = {n: load_run(OUT, f"01-{n}") for n in NAMES if (OUT / f"01-{n}" / "metrics.json").exists()}
plots.results_table(runs, leaderboard="transformers/leaderboard.csv").round(4)

In [ ]:
plots.plot_curves({n: runs[n] for n in ["sentences", "synthetic", "sent10k-syn"] if n in runs}, title="single sources vs a mix")

In [ ]:
plots.plot_curves({n: runs[n] for n in ["two-stage"] if n in runs}, title="two stages: sentences, then synthetic posts")

In [ ]:
for n in ["sentences", "sent10k-syn"]:
    if n in runs:
        plots.plot_grad_flow(runs[n], title=n)

## Conclusions

Choose by `half_B`; use `gold26` and the leaderboard as checks, and `cv` to see what each mix costs on
the sentence domain. Questions to answer here:

- How big is the gap between `cv` and `half_B` for sentences only, and how much do synthetic posts close it?
- Does `half_A` beat `half_B` by much? Then the model profits from the generator having seen half A,
  and `half_B` is the honest score.
- Is filtering by the adversarial score worth the lost data?

Then write the winning data setting into `ner/recipes.py` (`DATA`, and `STAGE2` if two stages win).